# MOFSemNet — train and SAVE the models for the live demo (server notebook)

Self-contained (no imports from the repository). Upload next to this notebook:
`MOFCSD.csv`, `chemberta_embeddings.npz` and `molt5_embeddings.npz`. Then **Run All**.

What it saves in `mofsemnet_models/` (and zips it as `mofsemnet_models.zip` at the end):

| folder | contents |
|---|---|
| `trees/` | tree models for both input modes (ExtraTrees, Random Forest, XGBoost, LightGBM, CatBoost) in their native formats |
| `gnns/` | every GNN of the grid as a PyTorch file: **all weights and biases** (`state_dict`) + its settings, scalers, graph file name and interval width |
| `graphs/` | every graph used (edge list, edge weights, which MOFs survived Black Hole pruning) |
| `features/` | feature column layout, metal order, the input and target scalers, the train / validation / test split |
| `results/` | `model_results.csv`: test R², MAE, RMSE, validation R² and the 90 % interval half-width of every saved model |
| `manifest.json`, `README.txt` | every file with size and SHA-256, the configuration, data hash and package versions |

Grid: 2 input modes (linker + metal + pore geometry / linker + metal only) × 5 graph types (fingerprint, ChemBERTa semantic, ChemBERTa hybrid,
MolT5 semantic, MolT5 hybrid) × k ∈ {3, 5, 10} × pruning {none, Black Hole 30 %, Black Hole 50 %} × 3 GNNs (GCN, GraphSAGE, residual GCN)
= **135 GNNs per mode** (the agreed 81 with ChemBERTa + 54 with MolT5), plus the same 3 GNNs with no edges and 5 tree models per mode. Settings are the ones chosen on the validation split in the tuning runs.

Every model is trained on the training split with early stopping on the validation split and scored once on the fixed test set
(random 15 %, `split_state = 42`, seed 0). It is **resumable**: re-running skips models whose file already exists.

## 1. Install / imports

In [1]:
# On a fresh server (PyTorch Geometric needs a torch build that matches your CUDA):
!pip install -q pandas numpy networkx scikit-learn rdkit tqdm joblib xgboost lightgbm catboost transformers
!pip install -q torch_geometric

import hashlib, json, os, platform, random, sys, time, warnings, zipfile
from datetime import datetime

import joblib
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from networkx.algorithms.community import louvain_communities
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.nn import GCNConv, SAGEConv
from tqdm.auto import tqdm

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
device: cuda | torch 2.9.1+cu128


## 2. Configuration  ← the only cell you normally edit

`PROFILE = "smoke"` runs a tiny version (800 MOFs, a few models) in a minute to check everything works; `"full"` is the real run.

In [2]:
PROFILE = "full"                       # "smoke" | "full"

DATA_PATH = "MOFCSD.csv"
EMB_PATHS = {"chemberta": "chemberta_embeddings.npz", "molt5": "molt5_embeddings.npz"}
HF_IDS = {"chemberta": "seyonec/ChemBERTa-zinc-base-v1", "molt5": "laituan245/molt5-base"}
OUT_DIR = "mofsemnet_models"

CFG = dict(
    n_subsample=None, seed=0, split_state=42, test_frac=0.15, val_frac=0.15,
    modes=["with_pore", "chemistry_only"],              # with_pore: linker + metal + 2 pore columns; chemistry_only: linker + metal
    graph_types=["fingerprint", "semantic_chemberta", "hybrid_chemberta", "semantic_molt5", "hybrid_molt5"],
    k_values=[3, 5, 10],
    pruning=["none", "black_hole_30", "black_hole_50"],
    gnn_models=["gcn", "graphsage", "residual_gcn"],
    include_no_edge=True,                               # also save the same GNNs with no edges (the reference row)
    tree_models=["extra_trees", "random_forest", "xgboost", "lightgbm", "catboost"],
    epochs=250, patience=25, conformal_alpha=0.10, hybrid_beta=0.5, bh_weights=(0.33, 0.33, 0.33),
)

# Settings chosen on the VALIDATION split in the tuning runs (scripts/tune_gnns.py, scripts/run_model_zoo.py)
GNN_SETTINGS = {   # used for every graph (fingerprint / semantic / hybrid, any k, any pruning)
    "gcn":          {"hidden": 256, "layers": 2, "dropout": 0.5, "lr": 0.01, "weight_decay": 1e-3},
    "graphsage":    {"hidden": 256, "layers": 2, "dropout": 0.5, "lr": 0.01, "weight_decay": 1e-3},
    "residual_gcn": {"hidden": 256, "layers": 1, "dropout": 0.5, "lr": 0.01, "weight_decay": 1e-3},
}
GNN_SETTINGS_NO_EDGE = {
    "gcn":          {"hidden": 128, "layers": 3, "dropout": 0.5, "lr": 0.002, "weight_decay": 1e-3},
    "graphsage":    {"hidden": 256, "layers": 2, "dropout": 0.5, "lr": 0.01, "weight_decay": 1e-3},
    "residual_gcn": {"hidden": 256, "layers": 1, "dropout": 0.5, "lr": 0.01, "weight_decay": 1e-3},
}
TREE_SETTINGS = {
    "extra_trees":   {"n_estimators": 300, "max_features": 0.3, "min_samples_leaf": 1},
    "random_forest": {"n_estimators": 300, "max_features": 0.3, "min_samples_leaf": 1},
    "xgboost":       {"max_depth": 10, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.3, "min_child_weight": 1, "reg_lambda": 5.0},
    "lightgbm":      {"num_leaves": 127, "learning_rate": 0.03, "colsample_bytree": 0.5, "subsample": 0.7, "min_child_samples": 5, "reg_lambda": 5.0},
    "catboost":      {"depth": 8, "learning_rate": 0.1, "l2_leaf_reg": 3.0},
}

if PROFILE == "smoke":
    CFG.update(n_subsample=800, graph_types=["fingerprint", "semantic_chemberta", "hybrid_molt5"], k_values=[3], pruning=["none", "black_hole_30"],
               gnn_models=["gcn", "residual_gcn"], tree_models=["random_forest", "xgboost"], epochs=20)
    TREE_SETTINGS = {k: dict(v, **({"n_estimators": 50} if "n_estimators" in v else {})) for k, v in TREE_SETTINGS.items()}
    OUT_DIR = "mofsemnet_models_smoke"

for sub in ("trees", "gnns", "graphs", "features", "results"):
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
n_gnn = len(CFG["modes"]) * (len(CFG["graph_types"]) * len(CFG["k_values"]) * len(CFG["pruning"]) + int(CFG["include_no_edge"])) * len(CFG["gnn_models"])
print(f"PROFILE={PROFILE} | GNNs to save: {n_gnn} | tree models: {len(CFG['modes']) * len(CFG['tree_models'])} | output: {OUT_DIR}/")

PROFILE=full | GNNs to save: 276 | tree models: 10 | output: mofsemnet_models/


## 3. Data and features (same code as the experiment notebooks)

In [3]:
INVALID_SMILES, INVALID_REPLACEMENT = "F[Si](F)(F)(F)(F)F", "c1ccccc1"
_MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def derive_pld_category(pld):
    if pd.isna(pld) or pld < 2.4:
        return "nonporous"
    if pld < 4.0:
        return "small pore"
    if pld < 8.0:
        return "medium pore"
    return "large pore"


def load_data(path, n_subsample=None):
    df = pd.read_csv(path, index_col=0)
    df["linker SMILES"] = df["linker SMILES"].replace(INVALID_SMILES, INVALID_REPLACEMENT)
    df = df.rename(columns={"linker SMILES": "linker_smiles", "Pore Limiting Diameter": "pld_value"})
    df["refcode"] = df.index
    df["pld_category"] = df["pld_value"].apply(derive_pld_category)
    if n_subsample:
        df = df.sample(n=min(n_subsample, len(df)), random_state=0)
    return df.reset_index(drop=True)


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def morgan_matrix(smiles_series):
    """1024-bit Morgan fingerprints (radius 2); invalid SMILES -> all-zero row. Computed once per
    distinct SMILES (only ~3,200 distinct linkers exist for 14,296 MOFs)."""
    cache = {}
    for smi in smiles_series.unique():
        mol = Chem.MolFromSmiles(smi)
        cache[smi] = (np.zeros(1024, np.float32) if mol is None or mol.GetNumAtoms() == 0
                      else _MORGAN.GetFingerprintAsNumPy(mol).astype(np.float32))
    return np.stack([cache[s] for s in smiles_series.values]).astype(np.float32)


def build_fingerprint_features(df, fp):
    """1024 Morgan bits + 2 pore-geometry columns + one-hot metal. PLD itself is NOT a feature."""
    pore = df[["Largest Cavity Diameter", "Largest Free Sphere"]].values.astype(np.float32)
    metals = sorted(df["metal"].dropna().unique().tolist())
    index = {m: i for i, m in enumerate(metals)}
    one_hot = np.zeros((len(df), len(metals)), np.float32)
    for row, m in enumerate(df["metal"].values):
        one_hot[row, index[m]] = 1.0
    return np.concatenate([fp, pore, one_hot], axis=1).astype(np.float32)


def embed_with_hf(smiles_list, lm, batch_size=64):
    from transformers import AutoModel, AutoTokenizer, T5EncoderModel
    tok = AutoTokenizer.from_pretrained(HF_IDS[lm])
    model = (T5EncoderModel if lm == "molt5" else AutoModel).from_pretrained(HF_IDS[lm]).to(DEVICE).eval()
    chunks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(smiles_list), batch_size), desc=f"embedding ({lm})"):
            enc = tok(smiles_list[i:i + batch_size], padding=True, truncation=True, return_tensors="pt").to(DEVICE)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            chunks.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)).cpu().numpy())
    return np.concatenate(chunks).astype(np.float32)


def get_embeddings(df, lm):
    """Looks every SMILES up in the uploaded .npz (keyed by SMILES text, so row order and
    subsampling do not matter); only SMILES missing from the file are computed with the model."""
    lookup = {}
    path = EMB_PATHS[lm]
    if os.path.exists(path):
        cached = np.load(path, allow_pickle=True)
        for smi, vec in zip(cached["smiles"], cached["embeddings"]):
            lookup.setdefault(str(smi), vec)
        print(f"[{lm}] loaded {len(lookup)} cached embeddings from {path}")
    missing = [s for s in df["linker_smiles"].unique() if s not in lookup]
    if missing:
        print(f"[{lm}] computing {len(missing)} missing embeddings with {HF_IDS[lm]} (needs internet)")
        for smi, vec in zip(missing, embed_with_hf(missing, lm)):
            lookup[smi] = vec
    return np.stack([lookup[s] for s in df["linker_smiles"].values]).astype(np.float32)


df = load_data(DATA_PATH, CFG["n_subsample"])
N = len(df)
LABELS = df["pld_value"].values.astype(np.float32)
DATA_SHA = file_sha256(DATA_PATH)
FP = morgan_matrix(df["linker_smiles"])
X_ALL = build_fingerprint_features(df, FP)                      # 1024 bits | 2 pore columns | metal one-hot
METALS = sorted(df["metal"].dropna().unique().tolist())
FEATURES = {"with_pore": X_ALL, "chemistry_only": np.hstack([X_ALL[:, :1024], X_ALL[:, 1026:]]).astype(np.float32)}
COLUMNS = {"with_pore": [f"fp_bit_{i}" for i in range(1024)] + ["largest_cavity_diameter", "largest_free_sphere"] + [f"metal_{m}" for m in METALS],
           "chemistry_only": [f"fp_bit_{i}" for i in range(1024)] + [f"metal_{m}" for m in METALS]}
LMS = sorted({g.split("_", 1)[1] for g in CFG["graph_types"] if "_" in g})
EMB = {lm: get_embeddings(df, lm) for lm in LMS}
print(f"MOFs={N}  metals={len(METALS)}  language models={LMS}  data sha256={DATA_SHA[:16]}")

[chemberta] loaded 3193 cached embeddings from chemberta_embeddings.npz
[molt5] loaded 3193 cached embeddings from molt5_embeddings.npz
MOFs=14296  metals=53  language models=['chemberta', 'molt5']  data sha256=e23a42e742099b4c


## 4. Split, scalers (saved)

In [4]:
rng = np.random.default_rng(CFG["split_state"])
TEST = np.sort(rng.choice(N, size=max(1, round(CFG["test_frac"] * N)), replace=False))
TEST_SET = set(TEST.tolist())
remaining = np.array([i for i in range(N) if i not in TEST_SET])
np.random.RandomState(CFG["seed"]).shuffle(remaining)
n_val = max(1, int(len(remaining) * CFG["val_frac"]))
VAL, TRAIN = np.sort(remaining[:n_val]), np.sort(remaining[n_val:])
NON_TEST = np.sort(remaining)


def standardise(X):
    """z-score the non-binary columns (pore geometry) with statistics from non-test MOFs; binary columns unchanged."""
    nb = ~((X == 0) | (X == 1)).all(axis=0)
    mu, sd = X[NON_TEST][:, nb].mean(0), X[NON_TEST][:, nb].std(0)
    sd = np.where(sd == 0, 1.0, sd)
    out = X.copy()
    out[:, nb] = (out[:, nb] - mu) / sd
    return out.astype(np.float32), {"columns": np.where(nb)[0].tolist(), "mean": mu.tolist(), "std": sd.tolist()}


XS, SCALERS = {}, {}
for mode in CFG["modes"]:
    XS[mode], SCALERS[mode] = standardise(FEATURES[mode])
    with open(f"{OUT_DIR}/features/scaler_{mode}.json", "w") as fh:
        json.dump({"input_scaler": SCALERS[mode], "columns": COLUMNS[mode]}, fh)
with open(f"{OUT_DIR}/features/feature_layout.json", "w") as fh:
    json.dump({"morgan": {"radius": 2, "bits": 1024}, "metals_in_one_hot_order": METALS, "pore_columns": ["Largest Cavity Diameter", "Largest Free Sphere"],
               "modes": {m: len(COLUMNS[m]) for m in CFG["modes"]}, "target": "Pore Limiting Diameter (Angstrom)"}, fh, indent=1)
with open(f"{OUT_DIR}/features/split.json", "w") as fh:
    json.dump({"split_state": CFG["split_state"], "seed": CFG["seed"], "train": TRAIN.tolist(), "val": VAL.tolist(), "test": TEST.tolist(),
               "refcodes": df["refcode"].tolist()}, fh)
pd.DataFrame({"refcode": df["refcode"], "linker_smiles": df["linker_smiles"], "metal": df["metal"], "pld": LABELS}).to_csv(f"{OUT_DIR}/features/mofs.csv", index=False)
print(f"train {len(TRAIN)} | validation {len(VAL)} | test {len(TEST)}")

train 10330 | validation 1822 | test 2144


## 5. Graphs and Black Hole pruning (same code as the experiment notebooks; saved to `graphs/`)

In [5]:
BLOCK = 1500


def _normalize(a):
    norms = np.linalg.norm(a, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return a / norms


def _tanimoto_block(rows):
    inter = FP[rows] @ FP.T
    union = FP[rows].sum(1)[:, None] + FP.sum(1)[None, :] - inter
    union[union == 0] = 1.0
    return inter / union


def knn_graph(sim_fn, k):
    """Undirected top-k graph from a blocked similarity function (never builds the N x N matrix)."""
    g = nx.Graph()
    g.add_nodes_from(range(N))
    kk = min(k, N - 1)
    for start in range(0, N, BLOCK):
        rows = slice(start, min(start + BLOCK, N))
        sim = sim_fn(rows)
        local = np.arange(sim.shape[0])
        sim[local, local + start] = -1.0
        top = np.argpartition(-sim, kk - 1, axis=1)[:, :kk]
        for r in range(sim.shape[0]):
            for c in top[r]:
                if sim[r, c] > -1.0:
                    g.add_edge(start + r, int(c), weight=float(sim[r, c]))
    return g


def edge_arrays(graph):
    edges = list(graph.edges(data=True))
    if not edges:
        return np.zeros((2, 0), np.int64), np.zeros(0, np.float32)
    src = [u for u, v, d in edges] + [v for u, v, d in edges]
    dst = [v for u, v, d in edges] + [u for u, v, d in edges]
    return np.array([src, dst]), np.array([d["weight"] for u, v, d in edges] * 2, np.float32)


def gravity_per_community(graph, communities, weights):
    wd, wb, ws = weights
    gravity, fixed_test = {}, set()
    for community in communities:
        sub = graph.subgraph(community)
        if sub.number_of_nodes() == 0:
            continue
        deg = nx.degree_centrality(sub)
        bet = nx.betweenness_centrality(sub, normalized=True)
        wsum = {n: sum(d["weight"] for _, _, d in sub.edges(n, data=True)) for n in sub.nodes()}
        nodes = list(community)
        norm = lambda vals: MinMaxScaler().fit_transform(np.array(vals).reshape(-1, 1)).flatten()
        nd, nb, nw = norm([deg[n] for n in nodes]), norm([bet[n] for n in nodes]), norm([wsum[n] for n in nodes])
        for i, n in enumerate(nodes):
            gravity[n] = wd * nd[i] + wb * nb[i] + ws * nw[i]
        degrees = {n: sub.degree(n) for n in community}
        eligible = [n for n, d in degrees.items() if d > 2] or ([max(degrees, key=degrees.get)] if degrees else [])
        if eligible:
            ranked = sorted(eligible, key=lambda n: gravity[n], reverse=True)
            fixed_test.update(ranked[:max(1, int(0.04 * len(community)))])
    return gravity, fixed_test


def black_hole_nodes(graph, gravity, communities, threshold, fixed_test, pld_categories):
    g = graph.copy()
    selected = set(fixed_test)
    n_total = g.number_of_nodes()
    target = int(max(0.2 * n_total, (1 - threshold) * n_total))
    target_non_test = max(0, target - len(fixed_test))
    sizes = [len(c) for c in communities]
    total = sum(sizes) or 1
    targets = [max(1, int(target_non_test * (s / total))) for s in sizes]
    diff = target_non_test - sum(targets)
    for i in range(abs(diff)):
        j = i % len(targets)
        targets[j] = max(0, targets[j] + (1 if diff > 0 else -1))
    remove = []
    for community, tcount in zip(communities, targets):
        nodes = [n for n in community if n in gravity and n not in fixed_test]
        if not nodes:
            continue
        props = pld_categories.iloc[nodes].value_counts(normalize=True).to_dict()
        for cat, prop in props.items():
            cand = [n for n in nodes if pld_categories.iloc[n] == cat]
            selected.update(sorted(cand, key=lambda n: gravity[n], reverse=True)[:max(1, int(tcount * prop))])
        remove.extend(n for n in nodes if n not in selected)
    g.remove_nodes_from(remove)
    return g


def prune_edges(graph, edge_threshold, fixed_test):
    edges = [(u, v, d["weight"]) for u, v, d in graph.edges(data=True)]
    if not edges:
        return graph
    test_edges = [e for e in edges if e[0] in fixed_test or e[1] in fixed_test]
    other = [e for e in edges if e[0] not in fixed_test and e[1] not in fixed_test]
    best = {}
    for u, v, w in test_edges:
        for n in (u, v):
            if n in fixed_test and (n not in best or w > best[n][2]):
                best[n] = (u, v, w)
    keep_test = list({(min(u, v), max(u, v)): (u, v, w) for u, v, w in best.values()}.values())
    n_keep = max(0, max(10, int((1 - edge_threshold) * len(edges))) - len(keep_test))
    other.sort(key=lambda e: e[2], reverse=True)
    out = nx.Graph()
    out.add_nodes_from(graph.nodes())
    out.add_weighted_edges_from(keep_test + other[:n_keep])
    return out


def restore_test_nodes(pruned, original, fixed_test):
    """Add back any test node pruning dropped (with its original edges) so every method is
    evaluated on the identical test set."""
    missing = fixed_test - set(pruned.nodes())
    if not missing:
        return pruned
    out = pruned.copy()
    out.add_nodes_from(missing)
    for node in missing:
        for nb, d in original.adj[node].items():
            if nb in out:
                out.add_edge(node, nb, **d)
    return out


def similarity_fn(graph_type):
    if graph_type == "fingerprint":
        return _tanimoto_block
    kind, lm = graph_type.split("_", 1)
    en = _normalize(EMB[lm])
    if kind == "semantic":
        return lambda r: en[r] @ en.T
    b = CFG["hybrid_beta"]
    return lambda r: b * (en[r] @ en.T) + (1 - b) * _tanimoto_block(r)


_BASE = {}


def get_graph(graph_type, k, pruning):
    """Returns (edge_index, edge_weight, kept_nodes) and saves them to graphs/<graph>_k<k>_<pruning>.npz (reused when present)."""
    path = f"{OUT_DIR}/graphs/{graph_type}_k{k}_{pruning}.npz"
    if os.path.exists(path):
        z = np.load(path)
        return z["edge_index"], z["edge_weight"], z["kept_nodes"]
    t = time.time()
    if (graph_type, k) not in _BASE:
        _BASE[(graph_type, k)] = knn_graph(similarity_fn(graph_type), k)
    base = _BASE[(graph_type, k)]
    if pruning == "none":
        g = base
    else:
        retention = int(pruning.split("_")[-1]) / 100
        communities = louvain_communities(base, seed=42)
        gravity, own_test = gravity_per_community(base, communities, CFG["bh_weights"])
        pruned = black_hole_nodes(base, gravity, communities, 1 - retention, own_test, df["pld_category"])
        pruned = prune_edges(pruned, 1 - retention, own_test)
        g = restore_test_nodes(pruned, base, TEST_SET)
    ei, ew = edge_arrays(g)
    kept = np.zeros(N, dtype=bool)
    kept[list(g.nodes())] = True
    np.savez_compressed(path, edge_index=ei, edge_weight=ew, kept_nodes=kept)
    print(f"graph {graph_type} k={k} {pruning}: {ei.shape[1] // 2} edges, {kept.sum()} MOFs kept ({time.time() - t:.0f}s)")
    return ei, ew, kept

## 6. Models

In [6]:
class Net(nn.Module):
    """GCN / GraphSAGE / residual GCN, as in the tuning runs (PyTorch Geometric layers)."""

    def __init__(self, kind, d_in, p):
        super().__init__()
        self.kind = kind
        h, L, self.drop = p["hidden"], p["layers"], p["dropout"]
        if kind == "gcn":
            self.convs = nn.ModuleList([GCNConv(d_in if i == 0 else h, 1 if i == L - 1 else h) for i in range(L)])
        elif kind == "graphsage":
            self.convs = nn.ModuleList([SAGEConv(d_in if i == 0 else h, 1 if i == L - 1 else h) for i in range(L)])
        elif kind == "residual_gcn":
            self.lin_in, self.lin_out = nn.Linear(d_in, h), nn.Linear(h, 1)
            self.blocks = nn.ModuleList([GCNConv(h, h) for _ in range(L)])
            self.gates = nn.Parameter(torch.zeros(L))
        else:
            raise ValueError(kind)

    def forward(self, x, ei, ew=None):
        if self.kind in ("gcn", "graphsage"):
            for conv in self.convs[:-1]:
                x = F.dropout(F.relu(conv(x, ei, ew) if self.kind == "gcn" else conv(x, ei)), p=self.drop, training=self.training)
            return self.convs[-1](x, ei, ew) if self.kind == "gcn" else self.convs[-1](x, ei)
        h = F.dropout(F.relu(self.lin_in(x)), p=self.drop, training=self.training)
        for gate, conv in zip(self.gates, self.blocks):
            h = h + gate * F.relu(conv(h, ei, ew))
        return self.lin_out(h)


def metrics(y, p):
    return {"r2": float(r2_score(y, p)), "mae": float(mean_absolute_error(y, p)), "rmse": float(np.sqrt(mean_squared_error(y, p)))}


def conformal_halfwidth(y_val, p_val, alpha):
    """Split-conformal half-width from validation residuals: prediction +/- q covers (1 - alpha) of new MOFs."""
    r = np.sort(np.abs(y_val - p_val))
    k = min(len(r) - 1, int(np.ceil((len(r) + 1) * (1 - alpha))) - 1)
    return float(r[k])


def set_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_gnn(kind, p, Xs, ei_np, ew_np, seed):
    set_seeds(seed)
    x = torch.tensor(Xs, device=DEVICE)
    ei = torch.tensor(ei_np, dtype=torch.long, device=DEVICE)
    ew = torch.tensor(ew_np, dtype=torch.float, device=DEVICE) if ew_np.size else None
    y_mean, y_std = float(LABELS[TRAIN].mean()), float(LABELS[TRAIN].std())
    y = torch.tensor((LABELS - y_mean) / y_std, device=DEVICE)
    tr, va = torch.tensor(TRAIN, device=DEVICE), torch.tensor(VAL, device=DEVICE)
    model = Net(kind, Xs.shape[1], p).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
    best, state, wait, ran = float("inf"), None, 0, 0
    for ran in range(1, CFG["epochs"] + 1):
        model.train(); opt.zero_grad()
        F.mse_loss(model(x, ei, ew).squeeze(-1)[tr], y[tr]).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = F.mse_loss(model(x, ei, ew).squeeze(-1)[va], y[va]).item()
        if vl < best:
            best, wait, state = vl, 0, {k_: v.detach().clone() for k_, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= CFG["patience"]:
                break
    model.load_state_dict(state); model.eval()
    with torch.no_grad():
        pred = model(x, ei, ew).squeeze(-1).cpu().numpy() * y_std + y_mean
    return model, {"y_mean": y_mean, "y_std": y_std, "epochs_run": ran, "best_val_mse_standardised": best}, pred


def make_tree(name, p, seed):
    if name == "extra_trees":
        return ExtraTreesRegressor(random_state=seed, n_jobs=-1, **p)
    if name == "random_forest":
        return RandomForestRegressor(random_state=seed, n_jobs=-1, **p)
    if name == "xgboost":
        import xgboost as xgb
        return xgb.XGBRegressor(n_estimators=2000, tree_method="hist", early_stopping_rounds=50, random_state=seed, n_jobs=-1, verbosity=0, **p)
    if name == "lightgbm":
        import lightgbm as lgb
        return lgb.LGBMRegressor(n_estimators=2000, subsample_freq=1, random_state=seed, n_jobs=-1, verbose=-1, **p)
    if name == "catboost":
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=1500, rsm=0.3, random_seed=seed, thread_count=-1, verbose=0, early_stopping_rounds=100, **p)
    raise ValueError(name)


def fit_tree(name, model, X):
    if name == "xgboost":
        model.fit(X[TRAIN], LABELS[TRAIN], eval_set=[(X[VAL], LABELS[VAL])], verbose=False)
    elif name == "lightgbm":
        import lightgbm as lgb
        model.fit(X[TRAIN], LABELS[TRAIN], eval_set=[(X[VAL], LABELS[VAL])], callbacks=[lgb.early_stopping(50, verbose=False)])
    elif name == "catboost":
        model.fit(X[TRAIN], LABELS[TRAIN], eval_set=(X[VAL], LABELS[VAL]))
    else:
        model.fit(X[TRAIN], LABELS[TRAIN])
    return model


def save_tree(name, model, path_base):
    if name in ("extra_trees", "random_forest"):
        path = path_base + ".joblib"; joblib.dump(model, path, compress=3)
    elif name == "xgboost":
        path = path_base + ".json"; model.save_model(path)
    elif name == "lightgbm":
        path = path_base + ".txt"; model.booster_.save_model(path)
    else:
        path = path_base + ".cbm"; model.save_model(path)
    return path


RESULTS_PATH = f"{OUT_DIR}/results/model_results.csv"
RESULT_COLUMNS = ["model_id", "family", "model", "mode", "graph", "k", "pruning", "file", "settings", "test_r2", "test_mae", "test_rmse",
                  "val_r2", "interval_halfwidth_90", "epochs_run", "n_parameters", "train_seconds", "n_train", "n_val", "n_test", "seed",
                  "data_sha256", "device", "saved_at"]


def record(row):
    pd.DataFrame([row]).reindex(columns=RESULT_COLUMNS).to_csv(RESULTS_PATH, mode="a", header=not os.path.exists(RESULTS_PATH), index=False)


def done_ids():
    return set(pd.read_csv(RESULTS_PATH)["model_id"]) if os.path.exists(RESULTS_PATH) else set()

## 7. Train and save the tree models (both modes)

In [7]:
DONE = done_ids()
for mode in CFG["modes"]:
    X = FEATURES[mode]
    for name in CFG["tree_models"]:
        mid = f"tree__{mode}__{name}"
        if mid in DONE:
            print("skip (already saved):", mid); continue
        t0 = time.time()
        model = fit_tree(name, make_tree(name, TREE_SETTINGS[name], CFG["seed"]), X)
        pred = model.predict(X)
        path = save_tree(name, model, f"{OUT_DIR}/trees/{mode}__{name}")
        m = metrics(LABELS[TEST], pred[TEST])
        record({"model_id": mid, "family": "tree", "model": name, "mode": mode, "graph": "none", "k": 0, "pruning": "none", "file": os.path.relpath(path, OUT_DIR),
                "settings": json.dumps(TREE_SETTINGS[name]), "test_r2": m["r2"], "test_mae": m["mae"], "test_rmse": m["rmse"],
                "val_r2": float(r2_score(LABELS[VAL], pred[VAL])), "interval_halfwidth_90": conformal_halfwidth(LABELS[VAL], pred[VAL], CFG["conformal_alpha"]),
                "epochs_run": None, "n_parameters": None, "train_seconds": time.time() - t0, "n_train": len(TRAIN), "n_val": len(VAL), "n_test": len(TEST),
                "seed": CFG["seed"], "data_sha256": DATA_SHA[:16], "device": "cpu", "saved_at": datetime.now().isoformat(timespec="seconds")})
        print(f"{mid}: test R2 {m['r2']:.4f}  ({time.time() - t0:.0f}s)  -> {path}")

tree__with_pore__extra_trees: test R2 0.9005  (8s)  -> mofsemnet_models/trees/with_pore__extra_trees.joblib
tree__with_pore__random_forest: test R2 0.8967  (5s)  -> mofsemnet_models/trees/with_pore__random_forest.joblib
tree__with_pore__xgboost: test R2 0.8964  (7s)  -> mofsemnet_models/trees/with_pore__xgboost.json
tree__with_pore__lightgbm: test R2 0.8945  (5s)  -> mofsemnet_models/trees/with_pore__lightgbm.txt
tree__with_pore__catboost: test R2 0.8939  (5s)  -> mofsemnet_models/trees/with_pore__catboost.cbm
tree__chemistry_only__extra_trees: test R2 0.5777  (6s)  -> mofsemnet_models/trees/chemistry_only__extra_trees.joblib
tree__chemistry_only__random_forest: test R2 0.6115  (4s)  -> mofsemnet_models/trees/chemistry_only__random_forest.joblib
tree__chemistry_only__xgboost: test R2 0.6037  (4s)  -> mofsemnet_models/trees/chemistry_only__xgboost.json
tree__chemistry_only__lightgbm: test R2 0.5975  (5s)  -> mofsemnet_models/trees/chemistry_only__lightgbm.txt
tree__chemistry_only__catbo

## 8. Train and save the GNN grid (weights and biases of every model)

In [8]:
DONE = done_ids()
jobs = []
for mode in CFG["modes"]:
    if CFG["include_no_edge"]:
        jobs += [(mode, "none", 0, "none", kind) for kind in CFG["gnn_models"]]
    for gt in CFG["graph_types"]:
        for k in CFG["k_values"]:
            for pr in CFG["pruning"]:
                jobs += [(mode, gt, k, pr, kind) for kind in CFG["gnn_models"]]
todo = [j for j in jobs if f"gnn__{j[0]}__{j[1]}__k{j[2]}__{j[3]}__{j[4]}" not in DONE]
print(f"GNN jobs: {len(jobs)} | already saved: {len(jobs) - len(todo)} | to do: {len(todo)}")

for mode, gt, k, pr, kind in tqdm(todo, desc="GNNs"):
    mid = f"gnn__{mode}__{gt}__k{k}__{pr}__{kind}"
    t0 = time.time()
    if gt == "none":
        ei, ew, kept = np.zeros((2, 0), dtype=np.int64), np.zeros(0, dtype=np.float32), np.ones(N, dtype=bool)
        graph_file, p = None, GNN_SETTINGS_NO_EDGE[kind]
    else:
        ei, ew, kept = get_graph(gt, k, pr)
        graph_file, p = f"graphs/{gt}_k{k}_{pr}.npz", GNN_SETTINGS[kind]
    model, info, pred = train_gnn(kind, p, XS[mode], ei, ew, CFG["seed"])
    m = metrics(LABELS[TEST], pred[TEST])
    half = conformal_halfwidth(LABELS[VAL], pred[VAL], CFG["conformal_alpha"])
    n_params = sum(t.numel() for t in model.parameters())
    path = f"{OUT_DIR}/gnns/{mode}__{gt}__k{k}__{pr}__{kind}.pt"
    torch.save({
        "state_dict": {k_: v.cpu() for k_, v in model.state_dict().items()},     # every weight and bias of the network
        "model": kind, "settings": p, "d_in": int(XS[mode].shape[1]), "mode": mode, "graph": gt, "k": k, "pruning": pr,
        "graph_file": graph_file, "input_scaler_file": f"features/scaler_{mode}.json",
        "target_scaler": {"mean": info["y_mean"], "std": info["y_std"]},
        "interval_halfwidth_90": half, "test_metrics": m, "epochs_run": info["epochs_run"],
        "seed": CFG["seed"], "split_state": CFG["split_state"], "data_sha256": DATA_SHA[:16],
        "how_to_load": "Net(model, d_in, settings).load_state_dict(state_dict); inputs = scaled features, output * std + mean = PLD (A)",
    }, path)
    record({"model_id": mid, "family": "gnn", "model": kind, "mode": mode, "graph": gt, "k": k, "pruning": pr, "file": os.path.relpath(path, OUT_DIR),
            "settings": json.dumps(p), "test_r2": m["r2"], "test_mae": m["mae"], "test_rmse": m["rmse"], "val_r2": float(r2_score(LABELS[VAL], pred[VAL])),
            "interval_halfwidth_90": half, "epochs_run": info["epochs_run"], "n_parameters": n_params, "train_seconds": time.time() - t0,
            "n_train": len(TRAIN), "n_val": len(VAL), "n_test": len(TEST), "seed": CFG["seed"], "data_sha256": DATA_SHA[:16], "device": str(DEVICE),
            "saved_at": datetime.now().isoformat(timespec="seconds")})
print("done")

GNN jobs: 276 | already saved: 0 | to do: 276


GNNs:   0%|          | 0/276 [00:00<?, ?it/s]

graph fingerprint k=3 none: 36576 edges, 14296 MOFs kept (3s)
graph fingerprint k=3 black_hole_30: 5758 edges, 5730 MOFs kept (6s)
graph fingerprint k=3 black_hole_50: 10249 edges, 7951 MOFs kept (6s)
graph fingerprint k=5 none: 59400 edges, 14296 MOFs kept (3s)
graph fingerprint k=5 black_hole_30: 9008 edges, 5635 MOFs kept (10s)
graph fingerprint k=5 black_hole_50: 16755 edges, 8028 MOFs kept (10s)
graph fingerprint k=10 none: 115294 edges, 14296 MOFs kept (3s)
graph fingerprint k=10 black_hole_30: 17222 edges, 5683 MOFs kept (20s)
graph fingerprint k=10 black_hole_50: 32005 edges, 8043 MOFs kept (21s)
graph semantic_chemberta k=3 none: 36249 edges, 14296 MOFs kept (2s)
graph semantic_chemberta k=3 black_hole_30: 5637 edges, 5786 MOFs kept (7s)
graph semantic_chemberta k=3 black_hole_50: 10173 edges, 8024 MOFs kept (7s)
graph semantic_chemberta k=5 none: 58905 edges, 14296 MOFs kept (3s)
graph semantic_chemberta k=5 black_hole_30: 8745 edges, 5665 MOFs kept (11s)
graph semantic_chemb

## 9. Check: reload saved models and confirm they give the same predictions

In [9]:
res = pd.read_csv(RESULTS_PATH)
checks = []
# one tree
row = res[res.family == "tree"].iloc[0]
X = FEATURES[row["mode"]]
f = os.path.join(OUT_DIR, row["file"])
if f.endswith(".joblib"):
    m2 = joblib.load(f); p2 = m2.predict(X[TEST])
elif f.endswith(".json"):
    import xgboost as xgb; m2 = xgb.XGBRegressor(); m2.load_model(f); p2 = m2.predict(X[TEST])
elif f.endswith(".txt"):
    import lightgbm as lgb; p2 = lgb.Booster(model_file=f).predict(X[TEST])
else:
    from catboost import CatBoostRegressor; m2 = CatBoostRegressor(); m2.load_model(f); p2 = m2.predict(X[TEST])
checks.append((row["model_id"], abs(r2_score(LABELS[TEST], p2) - row["test_r2"])))
# one GNN with a graph
row = res[(res.family == "gnn") & (res.graph != "none")].iloc[0]
ck = torch.load(os.path.join(OUT_DIR, row["file"]), map_location="cpu")
net = Net(ck["model"], ck["d_in"], ck["settings"]); net.load_state_dict(ck["state_dict"]); net.eval()
z = np.load(os.path.join(OUT_DIR, ck["graph_file"]))
ew = torch.tensor(z["edge_weight"]) if z["edge_weight"].size else None
with torch.no_grad():
    p3 = net(torch.tensor(XS[ck["mode"]]), torch.tensor(z["edge_index"], dtype=torch.long), ew).squeeze(-1).numpy() * ck["target_scaler"]["std"] + ck["target_scaler"]["mean"]
checks.append((row["model_id"], abs(r2_score(LABELS[TEST], p3[TEST]) - row["test_r2"])))
for mid, gap in checks:
    print(("OK " if gap < 1e-3 else "MISMATCH ") + f"{mid}: reloaded test R2 differs by {gap:.2e}")

OK tree__with_pore__extra_trees: reloaded test R2 differs by 0.00e+00
OK gnn__with_pore__fingerprint__k3__none__gcn: reloaded test R2 differs by 0.00e+00


## 10. Results table, manifest, README, zip — download `mofsemnet_models.zip`

In [ ]:
res = pd.read_csv(RESULTS_PATH)
pd.set_option("display.width", 200)
print("Best models per mode (test R2):")
print(res.sort_values("test_r2", ascending=False).groupby("mode").head(8)[["mode", "family", "model", "graph", "k", "pruning", "test_r2", "test_mae", "interval_halfwidth_90"]].to_string(index=False))
summary = (res[res.family == "gnn"].groupby(["mode", "graph", "pruning", "model"]).test_r2.mean().unstack("model").round(3))
summary.to_csv(f"{OUT_DIR}/results/gnn_grid_summary.csv")


def _pkg(name):
    try:
        return __import__(name).__version__
    except Exception:
        return "?"


files = []
for root, _, names in os.walk(OUT_DIR):
    for n in sorted(names):
        fp = os.path.join(root, n)
        if n in ("manifest.json",):
            continue
        files.append({"file": os.path.relpath(fp, OUT_DIR), "bytes": os.path.getsize(fp), "sha256": file_sha256(fp)[:16]})
manifest = {"created": datetime.now().isoformat(timespec="seconds"), "profile": PROFILE, "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
            "gnn_settings": GNN_SETTINGS, "gnn_settings_no_edge": GNN_SETTINGS_NO_EDGE, "tree_settings": TREE_SETTINGS,
            "data_file": DATA_PATH, "data_sha256": DATA_SHA, "n_mofs": N, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "python": sys.version.split()[0], "platform": platform.platform(),
            "packages": {p: _pkg(p) for p in ["torch", "torch_geometric", "sklearn", "xgboost", "lightgbm", "catboost", "networkx", "rdkit", "numpy", "pandas"]},
            "n_models": int(len(res)), "total_bytes": int(sum(f["bytes"] for f in files)), "files": files}
with open(f"{OUT_DIR}/manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=1, default=str)
with open(f"{OUT_DIR}/README.txt", "w") as fh:
    fh.write(f"""MOFSemNet saved models for the live demo
Created {manifest['created']} with profile {PROFILE}; data {DATA_PATH} (SHA-256 {DATA_SHA}); seed {CFG['seed']}, split_state {CFG['split_state']}.
{len(res)} models: {int((res.family == 'tree').sum())} tree models, {int((res.family == 'gnn').sum())} GNNs. Results: results/model_results.csv.
GNN files (gnns/*.pt) hold the full state_dict (all weights and biases) plus settings, the target scaler, the graph file and the 90% interval half-width.
Load a GNN: build Net(model, d_in, settings) as in this notebook, load_state_dict, feed features scaled with features/scaler_<mode>.json,
then multiply the output by the target std and add the mean. Graphs: graphs/<graph>_k<k>_<pruning>.npz (edge_index, edge_weight, kept_nodes).
Tree models: trees/<mode>__<model>.(joblib|json|txt|cbm). Interval: prediction +/- interval_halfwidth_90 covers about 90% of new MOFs.
""")
zip_path = "mofsemnet_models.zip" if PROFILE == "full" else "mofsemnet_models_smoke.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, names in os.walk(OUT_DIR):
        for n in names:
            zf.write(os.path.join(root, n), arcname=os.path.relpath(os.path.join(root, n), os.path.dirname(OUT_DIR) or "."))
print(f"{len(res)} models saved, {manifest['total_bytes'] / 1e6:.0f} MB. Download: {os.path.abspath(zip_path)}")

Best models per mode (test R2):
          mode family         model              graph  k pruning  test_r2  test_mae  interval_halfwidth_90
     with_pore   tree   extra_trees               none  0    none 0.900463  0.463008               1.109942
     with_pore   tree random_forest               none  0    none 0.896696  0.499787               1.256425
     with_pore   tree       xgboost               none  0    none 0.896416  0.521153               1.247211
     with_pore   tree      lightgbm               none  0    none 0.894550  0.542566               1.283343
     with_pore   tree      catboost               none  0    none 0.893901  0.549473               1.273877
     with_pore    gnn     graphsage semantic_chemberta  5    none 0.882914  0.619010               1.435804
     with_pore    gnn     graphsage     semantic_molt5  3    none 0.882579  0.621860               1.407210
     with_pore    gnn     graphsage semantic_chemberta 10    none 0.882398  0.618830               1.425